## Transforming the Archivo Eltit - Rosenfeld digital collection to RDF

Created in July-September 2026 for the Pontificia Universidad Católica de Chile by Gustavo Candela

This dataset represents the descriptive metadata from the Moving Image Archive catalogue, which is Scotland’s national collection of moving images.

Data format: metadata available as Dublin Core by means of an OAI-PMH server
Data source: https://archivospatrimoniales.uc.cl/handle/123456789/31557

### Preparation

Import the libraries required to explore the summary of each record included in the dataset

In [3]:
from rdflib import Graph, URIRef, Literal, Namespace
from rdflib.namespace import FOAF, RDF, DCTERMS, VOID, DC, SKOS, OWL
import pandas as pd
import datetime
import xml.etree.ElementTree as ET

### Transformation to RDF

*Note: The variable domain could be updated to the domain of the organisation (e.g., https://archivospatrimoniales.uc.cl/).

In [4]:
domain = 'https://example.org/'
domainLanguage = domain + 'language/'

# folder to store the RDF outputs
folder_name = 'chile'
file_name = 'chile'

First, we instantiate all the namespaces that we will use when defining the RDF data

In [5]:
g = Graph()
g.bind("foaf", FOAF)
g.bind("rdf", RDF)
g.bind("dcterms", DCTERMS)
g.bind("dc", DC)
g.bind("void", VOID)
g.bind("skos", SKOS)
g.bind("owl", OWL)

schema = Namespace("https://schema.org/")
g.bind("schema", schema)

viaf = Namespace("https://viaf.org/viaf/")
g.bind("viaf", viaf)

wd = Namespace("http://www.wikidata.org/entity/")
g.bind("wd", wd)

### We define the dataset Archivo Eltit - Rosenfeld

In [6]:
eltit = URIRef(domain + "dataset/eltit")
g.add((eltit, RDF.type, schema.Dataset))
g.add((eltit, schema.url, URIRef("https://archivospatrimoniales.uc.cl/handle/123456789/31557")))
g.add((eltit, schema.description, Literal("El Archivo Eltit-Rosenfeld es un fondo documental creado por la artista Lotty Rosenfeld y la escritora Diamela Eltit a fines de la década del 80 y principios de la del 90, para el rescate de los testimonios y la memoria del movimiento de mujeres por el derecho a voto en Chile")))
g.add((eltit, schema.name, Literal("Archivo Eltit - Rosenfeld")))
g.add((eltit, DC.title, Literal("Archivo Eltit - Rosenfeld")))
g.add((eltit, schema.license, URIRef('https://creativecommons.org/publicdomain/zero/1.0/')))
g.add((eltit, schema.address, Literal("Biblioteca de Humanidades, 3er piso - Vicuña Mackenna 4860, Macul")))
g.add((eltit, schema.email, Literal("archivosuc@uc.cl")))

now = datetime.datetime.now()
g.add((eltit, schema.dateCreated, Literal(str(now)[:10])))

<Graph identifier=Nf49483412b9c47b283ee09154d6ee4f4 (<class 'rdflib.graph.Graph'>)>

### We open the XML records retrieved from the OAI server:
https://archivospatrimoniales.uc.cl/oai/request?verb=ListRecords&metadataPrefix=oai_dc&set=com_123456789_31557

In [7]:
ns = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "oai_dc": "http://www.openarchives.org/OAI/2.0/oai_dc/",
    "dc": "http://purl.org/dc/elements/1.1/"
}

tree = ET.parse("../datos/archivospatrimoniales_set_31557.xml")
root = tree.getroot()

### We extract each record and transform it to RDF using schema.org as main ontology

In [8]:
for record in root.findall(".//oai:record", ns):
    dc = record.find(".//oai_dc:dc", ns)

    datos = {}

    for elem in dc:
        tag = elem.tag.split("}")[-1]
        datos.setdefault(tag, []).append(elem.text)

    #print("Nuevo registro")

    identifier = ""
    uri = ""
    title = ""
    description = "" 
    typeTxt = ""
    externalUrl = ""
    publishedDate = ""
    dateCreated = ""
    encodingFormat = ""
    
    for campo, valores in datos.items():
        #if campo == "title":
        #print(campo, "->", valores)

        if campo == "identifier":
            uri = valores[0]
            if len(valores) >1 :
               identifier = valores[0] 
               uri  = valores[1]
        elif campo == "format":
            encodingFormat = valores[0]
            if len(valores) >1 :
                encodingFormat = valores[1]
        elif campo == "title":
            title = valores[0]
        elif campo == "description":
            description = valores[0]
            if len(valores) > 1:
                externalUrl = valores[1]
        elif campo == "type":
            typeTxt = valores[0]
        elif campo == "date":
            publishedDate = valores[0]
            if len(valores) > 2:
                dateCreated = valores[len(valores)-1]
        else:
            pass

    classtype = 'https://schema.org/CreativeWork'
    classtypeImage = 'https://schema.org/ImageObject'
    classtypeVideo = 'https://schema.org/VideoObject'
    classtypeText = 'https://schema.org/Text'

    
    record = URIRef(uri.strip())
    g.add((record, RDF.type, URIRef(classtype)))
    if typeTxt == "Fotografía":
        g.add((record, RDF.type, URIRef(classtypeImage)))
    elif typeTxt == "Video":
        g.add((record, RDF.type, URIRef(classtypeVideo)))
    elif typeTxt == "Manuscrito" or typeTxt == "Documento":
        g.add((record, RDF.type, URIRef(classtypeText)))
    else:
        pass
    
    #g.add((record, schema.sourceOrganization, Literal(row['sourceOrganisation'])))
    g.add((record, schema.isPartOf, eltit))
    if identifier != "":
        g.add((record, schema.identifier, Literal(identifier)))
    g.add((record, schema.datePublished, Literal(publishedDate)))
    if dateCreated != "":
        g.add((record, schema.dateCreated, Literal(dateCreated)))
    g.add((record, schema.name, Literal(title)))
    g.add((record, schema.encodingFormat, Literal(encodingFormat)))
    g.add((record, schema.additionalType, Literal(typeTxt)))
    if externalUrl != "":
        g.add((record, schema.url, URIRef(externalUrl)))
    g.add((record, schema.abstract, Literal(description)))
    g.add((record, schema.license, URIRef('https://creativecommons.org/publicdomain/zero/1.0/')))

### We store the graph in the form of a ttl file

In [9]:
g.serialize(destination="../datos/output/"+folder_name+"/dataset_"+file_name+".ttl")

<Graph identifier=Nf49483412b9c47b283ee09154d6ee4f4 (<class 'rdflib.graph.Graph'>)>